# Data locality

 - Author: Elwin van 't Wout
 - Affiliation: Pontificia Universidad Católica de Chile
 - Date: 18-8-2023

Test the efficiency of Python with different memory access.

In [ ]:
import numpy as np

## Loop strides

Test the influence of the stride of the loop on the efficiency. First, create a large NumPy array with random numbers. Then, perform two different sums, with the same number of operators.

A loop with ```range(0,N,1)``` has elements 0, 1, 2, 3, ..., N-1.

A loop with ```range(0,N*s,s)``` has elements 0, s, 2s, 3s, ..., (N-1)s.


In [ ]:
N = 1000000
stride = 187

In [ ]:
%%time
a = np.random.rand(N)

In [ ]:
%%timeit

sum1 = 0.0
for n in range(0,N,1):
    sum1 += a[n]


In [ ]:
%%time
b = np.random.rand(N*stride)

In [ ]:
%%timeit

sum2 = 0.0
for n in range(0,N*stride,stride):
    sum2 += b[n]


This experiment shows two algorithms, each with the same number of the same operations: $N$ summations. However, the timing is different. This can only be due to different memory access.

The efficiency of a NumPy array is different than for a Python list because both store the data in diferent formats.

In [ ]:
%%time
c = [np.random.rand() for n in range(N*stride)]

In [ ]:
%%timeit

sum3 = 0.0
for n in range(0,N*stride,stride):
    sum3 += c[n]


In general, it is almost always more efficient to use Numpy functionality.

In [ ]:
%%timeit

sum4 = np.sum(a)


In [ ]:
%%timeit

sum5 = np.sum(b[::stride])


## Summing the elements of a multi-dimensional arrays

The elements of a multidimensional arrays are stored in memory as a one-dimensional ordering. Hence, the order of accessing the elements has an impact on the timing. Let us create a 3-dimensional tensor and sum all elements. Again, the different implementations all require exactly the same number of the same operators ($N^3$ summations) but the memory access is different.

In [ ]:
N = 250

a = np.random.rand(N,N,N)
b = np.random.rand(N,N,N)
c = np.random.rand(N,N,N)

print("Created a random array of shape",a.shape)

In [ ]:
%%timeit

sum1 = 0.0
for i in range(N):
    for j in range(N):
        for k in range(N):
            sum1 += a[i,j,k]


In [ ]:
%%timeit

sum2 = 0.0
for k in range(N):
    for j in range(N):
        for i in range(N):
            sum2 += b[i,j,k]


Instead of using a loop, it is more efficient to use NumPy functions that are based on optimised algorithms and implementations.

In [ ]:
%%timeit

sum3 = np.sum(c)


## Python broadcasting

NumPy has optimised implementations for many algorithms. For example, summing a constant value to all elements in an array can be done with ```a+=1```. Even though the variable ```a``` has size ```(n,)``` and ```1``` is a scalar, the algorithm works. NumPy uses *broadcasting* which means that (if possible) the summation is performed for all elements.

In [ ]:
N = 1000000

In [ ]:
%%timeit

a = np.zeros(N)
for n in range(N):
    a[n] += 1


In [ ]:
%%timeit

b = np.zeros(N)
b += 1


## Matrix-matrix multiplication

Perform a matrix-matrix multiplication with different types of memory access.

In [ ]:
n = 500

A = np.random.rand(n,n)
B = np.random.rand(n,n)

In [ ]:
%%timeit

C1 = np.zeros((n,n))
for i in range(n):
    for j in range(n):
        C1[i,j] = np.dot(A[i,:],B[:,j])


In [ ]:
%%timeit

C2 = np.zeros((n,n))
for j in range(n):
    for k in range(n):
        C2[:,j] += A[:,k]*B[k,j]


In [ ]:
%%timeit

C3 = np.zeros((n,n))
for i in range(n):
    for k in range(n):
        C3[i,:] += A[i,k]*B[k,:]


In [ ]:
%%timeit

C4 = A @ B


Consider a larger matrix size.

In [ ]:
import time

n = 2000
A = np.random.rand(n,n)
B = np.random.rand(n,n)


C1 = np.zeros((n,n))
time_start = time.time()
for i in range(n):
    for j in range(n):
        C1[i,j] = np.dot(A[i,:],B[:,j])
time_end = time.time()
print("Finished the loops with order (i,j,*) in",time_end-time_start,"seconds.")

C2 = np.zeros((n,n))
time_start = time.time()
for j in range(n):
    for k in range(n):
        C2[:,j] += A[:,k]*B[k,j]
time_end = time.time()
print("Finished the loops with order (j,k,*) in",time_end-time_start,"seconds.")

C3 = np.zeros((n,n))
time_start = time.time()
for i in range(n):
    for k in range(n):
        C3[i,:] += A[i,k]*B[k,:]
time_end = time.time()
print("Finished the loops with order (i,k,*) in",time_end-time_start,"seconds.")

C4 = np.zeros((n,n))
time_start = time.time()
C4 = A @ B
time_end = time.time()
print("Finished the Numpy algorithm in",time_end-time_start,"seconds.")


## Apending data

Appending data to a Python list is efficient, but Numpy appends arrays by copying the data.

In [ ]:
n = 100000

In [ ]:
%%timeit

my_list = []

for i in range(n):
    my_list.append(i)

In [ ]:
%%timeit

my_array1 = np.array([], dtype=int)

for i in range(n):
    my_array1 = np.append(my_array1, i)

In [ ]:
%%timeit

my_array2 = np.empty(n, dtype=int)

for i in range(n):
    my_array2[i] = i